# 04.3 — RNN building blocks

The Optimal architecture's encoder is a GRU; its classifier is a Deep LSTM. Recurrent layers process a *sequence* — here, the `W` windows of a trial — carrying a hidden state forward through time. This notebook demystifies GRU/LSTM in PyTorch: the shapes, the `batch_first` convention, the two-part return, and the intuition for what the gates buy you over a plain feedforward stack.

This notebook covers:

* Why a recurrent layer for windowed neural data.
* `nn.GRU` / `nn.LSTM`: input/output shapes and `batch_first`.
* The `(output, hidden)` return — sequence vs final state.
* GRU vs LSTM: the practical difference, and why this project uses both.

**Prerequisites:** [02.6 nn.Module](../02_numpy_and_pytorch_basics/02.6_nn_module_vs_layergraph.ipynb), [04.2 the simple encoder](04.2_building_a_simple_encoder.ipynb).

## Section 1 — What MATLAB does

```matlab
gruLayer(HiddenSize, 'OutputMode', 'sequence')     % output at EVERY timestep
gruLayer(HiddenSize, 'OutputMode', 'last')         % output only the final timestep
lstmLayer(HiddenSize, 'OutputMode', 'sequence')
```

MATLAB expects data in `'CTB'` / `'SSCTB'` format — the time axis is labeled `T`, and the layer decides `sequence` vs `last` via `OutputMode`. Hidden-state management is automatic; you rarely touch it.

PyTorch is lower-level: you pick the axis order (via `batch_first`), and the layer *always* returns both the full sequence and the final hidden state — you slice whichever you need. Same recurrence, more explicit.

## Section 2 — The Python concepts you need

### 2.1 — Why recurrent at all

A trial is a *sequence* of windows ([02.2](../02_numpy_and_pytorch_basics/02.2_array_axis_conventions.ipynb): the `W` axis). A feedforward layer treats each window independently; a recurrent layer lets window *k*'s representation depend on windows 1…*k* — it accumulates context over the trial. For neural decoding, that temporal context is signal.

### 2.2 — `nn.GRU`: shapes and `batch_first`

In [ ]:
import torch
from torch import nn

gru = nn.GRU(input_size=8, hidden_size=16, batch_first=True)

x = torch.randn(4, 10, 8)          # (batch=4, sequence=10, features=8)  — batch FIRST
output, h_n = gru(x)
print("input :", tuple(x.shape),      "(batch, seq, features)")
print("output:", tuple(output.shape), "(batch, seq, hidden)   ← per-timestep")
print("h_n   :", tuple(h_n.shape),    "(num_layers, batch, hidden) ← final state only")

**`batch_first=True` is the single most important flag.** PyTorch's RNNs default to `batch_first=False` — expecting `(seq, batch, features)`, batch in the MIDDLE. That contradicts the batch-first convention everything else uses ([02.2 §2.1](../02_numpy_and_pytorch_basics/02.2_array_axis_conventions.ipynb)), so this project sets `batch_first=True` everywhere. Forget it, and your `(batch, seq, features)` tensor is silently interpreted as `(seq, batch, features)` — no error, just a scrambled batch/sequence axis and a model that can't learn.

| | MATLAB | PyTorch (this project) |
|---|---|---|
| axis order | `'CTB'` (labeled) | `(batch, seq, features)` via `batch_first=True` |
| sequence output | `OutputMode='sequence'` | `output` (the `[0]` return) |
| last-step output | `OutputMode='last'` | `output[:, -1]` or `h_n[-1]` |

### 2.3 — The two-part return

In [ ]:
# `output` holds EVERY timestep; `h_n` is just the last. They agree at the end:
last_from_output = output[:, -1]     # (batch, hidden) — final timestep of the sequence
last_from_state  = h_n[-1]           # (batch, hidden) — final hidden state
print("output[:, -1] == h_n[-1] ?", torch.allclose(last_from_output, last_from_state))

The encoder ([04.2](04.2_building_a_simple_encoder.ipynb)) keeps `output` (`[0]`) — the whole sequence, one representation per window, because the VAE reconstructs and classifies per-window. A classifier that only needs a per-trial verdict might instead take `output[:, -1]` (last window) — you'll see both patterns.

### 2.4 — GRU vs LSTM

In [ ]:
gru  = nn.GRU (input_size=8, hidden_size=16, batch_first=True)
lstm = nn.LSTM(input_size=8, hidden_size=16, batch_first=True)
print("GRU  params:", sum(p.numel() for p in gru.parameters()))
print("LSTM params:", sum(p.numel() for p in lstm.parameters()), " (~33% more — extra gate + cell state)")

# LSTM returns TWO states: (h_n, c_n) — hidden AND cell:
_, (h_n, c_n) = lstm(x)
print("LSTM state: h_n", tuple(h_n.shape), "+ c_n", tuple(c_n.shape))

Both are gated RNNs that solve the vanishing-gradient problem of plain RNNs; the practical differences:

| | GRU | LSTM |
|---|---|---|
| gates | 2 (reset, update) | 3 (input, forget, output) |
| extra state | none | a separate **cell** state (`c_n`) |
| parameters | fewer (~⅔ of LSTM) | more |
| return | `(output, h_n)` | `(output, (h_n, c_n))` — note the TUPLE |

This project uses a **GRU encoder** (fewer params, fast, plenty of capacity for the encoding job) and an **LSTM classifier** (the extra cell state helps the multi-step per-dimension decoding). The `(h_n, c_n)` tuple return is a common trip-up — LSTM's second element is itself a 2-tuple, so `output, hidden = lstm(x)` gives you `hidden = (h_n, c_n)`, not `h_n`.

## Section 3 — The `neural_data_decoding` implementation

The GRU/LSTM construction in `_EncoderBlock` — `batch_first=True` and the `[0]` sequence-output slice:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/models/encoder.py").read_text().splitlines()
i = next(j for j, line in enumerate(src) if 'elif transform == "GRU"' in line)
for k in range(i, min(i + 14, len(src))):
    print(f"{k + 1:4} | {src[k]}")

Things to spot:

* `batch_first=True` on both GRU and LSTM — the §2.2 convention, non-negotiable for this codebase's axis order.
* `input_size=in_features, hidden_size=hidden_size` — the block's in/out dims, chained across the stack ([04.2 §2.3](04.2_building_a_simple_encoder.ipynb)).
* The forward's `self.transform_layer(x)[0]` (seen in 04.2) discards the hidden state and keeps the sequence — MATLAB's `OutputMode='sequence'`.

## Section 4 — Hands-on exercises

### Exercise 4.1 — the batch_first trap, measured

Feed the SAME `(4, 10, 8)` tensor to a `batch_first=True` GRU and a `batch_first=False` GRU. Compare the output shapes and explain what the second one *thinks* the axes are.

In [ ]:
# Your attempt here


In [ ]:
# Reveal:
x = torch.randn(4, 10, 8)
g_true  = nn.GRU(8, 16, batch_first=True)
g_false = nn.GRU(8, 16, batch_first=False)
print("batch_first=True :", tuple(g_true(x)[0].shape),  "→ (batch=4, seq=10, hidden=16) ✓")
print("batch_first=False:", tuple(g_false(x)[0].shape), "→ reads it as (seq=4, batch=10, hidden=16) ✗")
print("Same input, silently different meaning — no error to warn you.")

### Exercise 4.2 — unpack LSTM correctly

Write the line that gets the LSTM's *cell* state from a forward call, then confirm its shape matches the hidden state.

In [ ]:
# Reveal:
lstm = nn.LSTM(8, 16, batch_first=True)
output, (h_n, c_n) = lstm(torch.randn(4, 10, 8))   # note the (h_n, c_n) tuple
print("cell state c_n:", tuple(c_n.shape), "== hidden h_n:", tuple(h_n.shape))

## Section 5 — Common errors

### Model won't learn; loss barely moves

Suspect `batch_first` (§2.2 / Exercise 4.1) — a batch-first tensor into a `batch_first=False` RNN scrambles the batch and sequence axes. The tell: output shape has your batch size where the sequence length should be.

### `ValueError: too many values to unpack` from an LSTM

`output, h_n = lstm(x)` fails because LSTM returns `(output, (h_n, c_n))`. Unpack as `output, (h_n, c_n) = lstm(x)`.

### `RuntimeError: input.size(-1) must be equal to input_size`

The RNN's `input_size` doesn't match the last axis of the input — the flattened-feature mismatch from [02.2 §5](../02_numpy_and_pytorch_basics/02.2_array_axis_conventions.ipynb). Check what the composite's FlattenPerWindow hands the encoder.

### Taking `h_n` when you wanted the sequence (or vice versa)

`output` is per-timestep; `h_n`/`h_n[-1]` is the final step only. Reconstruction/per-window tasks need `output`; a single per-trial verdict can use the last step.

### GRU and LSTM give different results with "the same" config

Expected — different gate counts and (for LSTM) a cell state make them different functions. They're not drop-in swaps; each wants its own tuned hyperparameters.

## Section 6 — Further reading

- [Understanding LSTMs (colah's blog)](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) — the canonical gate-by-gate intuition, with GRU at the end.
- [nn.GRU](https://pytorch.org/docs/stable/generated/torch.nn.GRU.html) / [nn.LSTM](https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html) — the shape contracts.
- [`src/neural_data_decoding/models/classifier.py`](../../src/neural_data_decoding/models/classifier.py) — `DeepLSTMClassifier`, the LSTM in its classifier role (next explored in [04.6](04.6_multi_head_classifier.ipynb)).

Next up: **[04.4 convolutional backbones](04.4_convolutional_backbones.ipynb)** — the Conv/Resnet/Multi-Filter encoders (Milestone CC.1) and how they plug into the same composite slot.